# LAB 3 — Checkpoint Recovery

This notebook demonstrates incremental restart using the same checkpoint and a safe full replay using a new checkpoint and a new target table.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Define isolated recovery-test resources

In [0]:
recovery_source_path = (
    f"{volume_root}/recovery_test/source"
)
same_checkpoint_path = (
    f"{volume_root}/system/checkpoints/recovery_same"
)
same_checkpoint_schema_path = (
    f"{volume_root}/system/schema/recovery_same"
)
same_checkpoint_table = (
    f"{catalog}.{schema}.lab03_checkpoint_recovery_same"
)

replay_checkpoint_path = (
    f"{volume_root}/system/checkpoints/recovery_replay"
)
replay_schema_path = (
    f"{volume_root}/system/schema/recovery_replay"
)
replay_table = (
    f"{catalog}.{schema}.lab03_checkpoint_recovery_replay"
)

print(f"Recovery source: {recovery_source_path}")
print(f"Same-checkpoint table: {same_checkpoint_table}")
print(f"Replay table: {replay_table}")

## 3. Reset the isolated recovery test

In [0]:
for path in [
    recovery_source_path,
    same_checkpoint_path,
    same_checkpoint_schema_path,
    replay_checkpoint_path,
    replay_schema_path,
]:
    dbutils.fs.rm(path, recurse=True)

dbutils.fs.mkdirs(recovery_source_path)

spark.sql(
    f"DROP TABLE IF EXISTS {same_checkpoint_table}"
)
spark.sql(
    f"DROP TABLE IF EXISTS {replay_table}"
)

print("Recovery test reset completed.")

## 4. Select source files for the test

In [0]:
available_initial_files = sorted(
    [
        file_info
        for file_info in dbutils.fs.ls(staging_initial_path)
        if file_info.name.endswith(".json")
    ],
    key=lambda file_info: file_info.name
)

if len(available_initial_files) < 30:
    raise RuntimeError(
        "At least 30 initial JSON files are required "
        "for the checkpoint recovery test."
    )

first_batch_files = available_initial_files[:20]
second_batch_files = available_initial_files[20:30]

print(f"First batch files: {len(first_batch_files)}")
print(f"Second batch files: {len(second_batch_files)}")

## 5. Define copy and ingestion helpers

In [0]:
from pyspark.sql.functions import col, current_timestamp

def copy_files(
    files: list,
    target_directory: str,
    prefix: str
) -> int:
    copied = 0

    for file_info in files:
        destination = (
            f"{target_directory}/{prefix}_{file_info.name}"
        )

        dbutils.fs.cp(
            file_info.path,
            destination
        )
        copied += 1

    return copied


def run_recovery_stream(
    source_path: str,
    schema_path: str,
    checkpoint_path: str,
    target_table: str,
    query_name: str
):
    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_path)
        .withColumn(
            "_source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "_source_file_name",
            col("_metadata.file_name")
        )
        .withColumn(
            "_ingested_at",
            current_timestamp()
        )
    )

    query = (
        stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .trigger(availableNow=True)
        .queryName(query_name)
        .toTable(target_table)
    )

    query.awaitTermination()
    return query

## 6. Initial run with the first batch

In [0]:
copied_first_batch = copy_files(
    first_batch_files,
    recovery_source_path,
    "batch1"
)

print(f"Copied first batch: {copied_first_batch}")

first_query = run_recovery_stream(
    source_path=recovery_source_path,
    schema_path=same_checkpoint_schema_path,
    checkpoint_path=same_checkpoint_path,
    target_table=same_checkpoint_table,
    query_name="lab03_recovery_first_run"
)

first_run_df = spark.table(same_checkpoint_table)

first_run_files = (
    first_run_df
    .select("_source_file")
    .distinct()
    .count()
)
first_run_rows = first_run_df.count()

print(f"First-run files: {first_run_files}")
print(f"First-run rows: {first_run_rows:,}")

## 7. Restart with the same checkpoint after adding new files

In [0]:
copied_second_batch = copy_files(
    second_batch_files,
    recovery_source_path,
    "batch2"
)

print(f"Copied second batch: {copied_second_batch}")

second_query = run_recovery_stream(
    source_path=recovery_source_path,
    schema_path=same_checkpoint_schema_path,
    checkpoint_path=same_checkpoint_path,
    target_table=same_checkpoint_table,
    query_name="lab03_recovery_second_run"
)

second_run_df = spark.table(same_checkpoint_table)

second_run_files = (
    second_run_df
    .select("_source_file")
    .distinct()
    .count()
)
second_run_rows = second_run_df.count()

new_files_processed = (
    second_run_files - first_run_files
)
new_rows_processed = (
    second_run_rows - first_run_rows
)

print(f"Files after restart: {second_run_files}")
print(f"New files processed: {new_files_processed}")
print(f"New rows processed: {new_rows_processed:,}")

## 8. Validate same-checkpoint recovery

In [0]:
expected_total_files = (
    copied_first_batch + copied_second_batch
)

assert first_run_files == copied_first_batch, (
    "The first run did not process the expected file count."
)

assert new_files_processed == copied_second_batch, (
    "The second run did not process only the newly added files."
)

assert second_run_files == expected_total_files, (
    "The final processed-file count is incorrect."
)

same_checkpoint_result_df = spark.createDataFrame(
    [
        ("First run files", first_run_files),
        ("Second batch files added", copied_second_batch),
        ("New files processed after restart", new_files_processed),
        ("Final processed files", second_run_files),
    ],
    ["metric", "value"]
)

display(same_checkpoint_result_df)

print(
    "Same-checkpoint recovery succeeded: previously processed files "
    "were skipped and only new files were ingested."
)

## 9. Perform a safe replay with a new checkpoint

The replay writes to a new target table. This avoids duplicating data in the original recovery table.

In [0]:
replay_query = run_recovery_stream(
    source_path=recovery_source_path,
    schema_path=replay_schema_path,
    checkpoint_path=replay_checkpoint_path,
    target_table=replay_table,
    query_name="lab03_recovery_safe_replay"
)

replay_df = spark.table(replay_table)

replay_files = (
    replay_df
    .select("_source_file")
    .distinct()
    .count()
)
replay_rows = replay_df.count()

print(f"Replay files: {replay_files}")
print(f"Replay rows: {replay_rows:,}")

## 10. Compare incremental recovery with safe replay

In [0]:
comparison_df = spark.createDataFrame(
    [
        (
            "Same checkpoint",
            same_checkpoint_table,
            second_run_files,
            second_run_rows,
            "Resumed and processed only new files"
        ),
        (
            "New checkpoint",
            replay_table,
            replay_files,
            replay_rows,
            "Reprocessed every file into a new table"
        ),
    ],
    [
        "test",
        "target_table",
        "processed_files",
        "output_rows",
        "behavior",
    ]
)

display(comparison_df)

assert replay_files == second_run_files, (
    "Replay did not process all source files."
)

assert replay_rows == second_run_rows, (
    "Replay row count does not match the recovered table."
)

## 11. Inspect checkpoint directories

In [0]:
print("Same-checkpoint contents:")
display(dbutils.fs.ls(same_checkpoint_path))

print("Replay-checkpoint contents:")
display(dbutils.fs.ls(replay_checkpoint_path))

## 12. Final result

In [0]:
print("Checkpoint recovery test completed successfully.")
print(
    "Same checkpoint: incremental resume without reprocessing old files."
)
print(
    "New checkpoint + new table: safe full replay of all source files."
)
print("Next notebook: lab03_06_eventhub_producer")